# Model Optimization: Pruning

In this notebook, we'll apply pruning techniques to our models using distributed processing.

In [ ]:
import json
import time
import pandas as pd
import boto3
import sagemaker
import os
import torch
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.pytorch.processing import PyTorchProcessor
from IPython.display import clear_output
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import AutoModelForTokenClassification, AutoModelForQuestionAnswering
from transformers import AutoModelForMaskedLM

## 1. Load Workshop Settings

In [ ]:
# Load stored variables
%store -r S3_BUCKET
%store -r AWS_REGION
%store -r SAGEMAKER_ROLE_ARN
%store -r OPTIMIZATION_INSTANCE_TYPE

# Check if variables were successfully retrieved
if 'S3_BUCKET' in locals() and S3_BUCKET != "YOUR_BUCKET_NAME_HERE":
    print("Workshop settings loaded successfully:")
    print(f"S3 Bucket: {S3_BUCKET}")
    print(f"AWS Region: {AWS_REGION}")
    print(f"SageMaker Role ARN: {SAGEMAKER_ROLE_ARN}")
    print(f"Optimization Instance Type: {OPTIMIZATION_INSTANCE_TYPE}")
else:
    print("⚠️ Workshop settings not found or not configured.")
    print("Please run the first notebook (01_introduction_and_setup.ipynb) to configure settings.")

# Initialize S3 client
s3_client = boto3.client('s3')

## 2. Load Model Information

In [ ]:
# Load model information from previous notebooks
try:
    with open('model_info.json', 'r') as f:
        model_info_dict = json.load(f)
    print(f"Loaded model information for {len(model_info_dict)} models")
except FileNotFoundError:
    print("model_info.json not found. Creating default model info.")
    model_info_dict = {
        "sentiment-analysis": {
            "model_name": "distilbert-base-uncased-finetuned-sst-2-english",
            "task": "text-classification",
            "hub_model_id": "distilbert-base-uncased-finetuned-sst-2-english",
            "s3_uri": f"s3://{S3_BUCKET}/models/distilbert-base-uncased-finetuned-sst-2-english"
        }
    }
    
    # Save model info to file
    with open('model_info.json', 'w') as f:
        json.dump(model_info_dict, f, indent=2)
    print("Created default model info with sentiment analysis model")

# Display model info
for model_key, info in model_info_dict.items():
    print(f"\nModel: {model_key}")
    print(f"  Name: {info['model_name']}")
    print(f"  Task: {info['task']}")
    print(f"  S3 URI: {info.get('s3_uri', 'Not available')}")

## 3. Configure Pruning Jobs

In [ ]:
# Define the instance type to use for pruning
instance_type = OPTIMIZATION_INSTANCE_TYPE
print(f"Using instance type: {instance_type} for optimization jobs")

# Create a SageMaker session
sagemaker_session = sagemaker.Session()

# Create a PyTorch processor
processor = PyTorchProcessor(
    framework_version="2.0.0",
    py_version="py310",
    role=SAGEMAKER_ROLE_ARN,
    instance_type=instance_type,
    instance_count=1,
    base_job_name="model-pruning",
    sagemaker_session=sagemaker_session
)

In [ ]:
# Launch pruning jobs for all models in parallel
job_names = []  # List to store all job names
job_output_paths = {}
s3_client = boto3.client('s3')

# First, prepare all the job configurations
job_configs = {}
print("Preparing pruning jobs for all models...")

for model_key in model_info_dict.keys():
    # Save model info to a temporary file
    with open(f'temp_{model_key}_info.json', 'w') as f:
        json.dump({model_key: model_info_dict[model_key]}, f)
    
    # Upload to S3
    s3_client.upload_file(
        f'temp_{model_key}_info.json', 
        S3_BUCKET, 
        f'optimization/inputs/{model_key}/model_info.json'
    )
    
    # Define the output path
    output_path = f's3://{S3_BUCKET}/optimization/outputs/{model_key}-pruned'
    job_output_paths[model_key] = output_path
    
    # Define inputs and outputs
    inputs = [
        ProcessingInput(
            source=f's3://{S3_BUCKET}/optimization/inputs/{model_key}/model_info.json',
            destination='/opt/ml/processing/input/data'
        )
    ]
    
    outputs = [
        ProcessingOutput(
            output_name='pruned-model',
            source='/opt/ml/processing/output',
            destination=output_path
        )
    ]
    
    # Store the job configuration
    job_configs[model_key] = {
        'inputs': inputs,
        'outputs': outputs,
        'arguments': [
            '--model-info-path', '/opt/ml/processing/input/data/model_info.json',
            '--output-dir', '/opt/ml/processing/output',
            '--pruning-method', 'structured',
            '--pruning-amount', '0.3'
        ],
        'output_path': output_path
    }
    print(f"Prepared job configuration for {model_key}")

# Now launch all jobs in parallel
print("\nLaunching all pruning jobs in parallel...")
for model_key, config in job_configs.items():
    try:
        # Create a unique job name with timestamp to avoid conflicts
        timestamp = int(time.time())
        job_name = f"pruning-{model_key}-{timestamp}"
        
        # Run the processing job with the unique name
        processor.run(
            code='pruning_script.py',
            source_dir='pruning_scripts',
            inputs=config['inputs'],
            outputs=config['outputs'],
            arguments=config['arguments'],
            wait=False,
            job_name=job_name
        )
        
        # Store the job name for tracking
        job_names.append(job_name)
        print(f"Launched job for {model_key}: {job_name}")
    except Exception as e:
        print(f"Error launching job for {model_key}: {e}")

print("\nAll jobs launched. You can monitor their progress in the SageMaker console.")

## 4. Analyze Results

In [ ]:
# Define functions for metrics collection
def get_model_size(model):
    """Calculate model size in MB."""
    param_size = 0
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    buffer_size = 0
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()
    
    size_mb = (param_size + buffer_size) / 1024**2
    return size_mb

def get_num_parameters(model):
    """Calculate number of parameters in the model."""
    return sum(p.numel() for p in model.parameters())

def count_non_zero_params(model):
    """Count non-zero parameters in the model."""
    non_zero = 0
    total = 0
    for param in model.parameters():
        if param.dim() > 1:  # Only count weights, not biases
            non_zero += torch.count_nonzero(param).item()
            total += param.numel()
    return non_zero, total

def prepare_sample_inputs(model_name, task, tokenizer, device):
    """Prepare sample inputs for the model based on its task."""
    if task == "sequence-classification" or task == "text-classification":
        text = "I really enjoyed this movie. The acting was superb and the plot was engaging."
        inputs = tokenizer(text, return_tensors="pt")
    elif task == "token-classification":
        text = "Jeff Bezos founded Amazon in 1994 and the company is headquartered in Seattle, Washington."
        inputs = tokenizer(text, return_tensors="pt")
    elif task == "question-answering":
        question = "What is machine learning?"
        context = "Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data."
        inputs = tokenizer(question, context, return_tensors="pt")
    elif task == "masked-lm" or task == "fill-mask":
        text = "The [MASK] is a large language model trained by OpenAI."
        inputs = tokenizer(text, return_tensors="pt")
    else:
        raise ValueError(f"Unsupported task: {task}")
    
    # Move inputs to the appropriate device
    return {k: v.to(device) for k, v in inputs.items()}

In [ ]:
# Collect metrics for all pruned models
all_metrics = {}

for model_key, job_info in job_configs.items():
    print(f"\nAnalyzing pruned model: {model_key}")
    
    # Get model info
    model_info = model_info_dict[model_key]
    model_name = model_info["model_name"]
    task = model_info["task"]
    
    # Set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Load original model
    print(f"Loading original model: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    if task == "sequence-classification" or task == "text-classification":
        original_model = AutoModelForSequenceClassification.from_pretrained(model_name)
    elif task == "token-classification":
        original_model = AutoModelForTokenClassification.from_pretrained(model_name)
    elif task == "question-answering":
        original_model = AutoModelForQuestionAnswering.from_pretrained(model_name)
    elif task == "masked-lm" or task == "fill-mask":
        original_model = AutoModelForMaskedLM.from_pretrained(model_name)
    else:
        raise ValueError(f"Unsupported task: {task}")
    
    original_model = original_model.to(device)
    original_model.eval()
    
    # Prepare sample inputs
    inputs = prepare_sample_inputs(model_name, task, tokenizer, device)
    
    # Measure baseline metrics
    baseline_size = get_model_size(original_model)
    baseline_params = get_num_parameters(original_model)
    baseline_non_zero, baseline_total = count_non_zero_params(original_model)
    
    print(f"Original model size: {baseline_size:.2f} MB")
    print(f"Original parameters: {baseline_params:,}")
    print(f"Original non-zero weights: {baseline_non_zero:,}/{baseline_total:,} ({baseline_non_zero/baseline_total*100:.2f}%)")
    
    # Load pruned model
    pruned_model_path = os.path.join(job_info["output_path"], f"{model_key}_pruned")
    print(f"Loading pruned model from: {pruned_model_path}")
    
    try:
        if task == "sequence-classification" or task == "text-classification":
            pruned_model = AutoModelForSequenceClassification.from_pretrained(pruned_model_path)
        elif task == "token-classification":
            pruned_model = AutoModelForTokenClassification.from_pretrained(pruned_model_path)
        elif task == "question-answering":
            pruned_model = AutoModelForQuestionAnswering.from_pretrained(pruned_model_path)
        elif task == "masked-lm" or task == "fill-mask":
            pruned_model = AutoModelForMaskedLM.from_pretrained(pruned_model_path)
        
        pruned_model = pruned_model.to(device)
        pruned_model.eval()
        
        # Measure pruned metrics
        pruned_size = get_model_size(pruned_model)
        pruned_params = get_num_parameters(pruned_model)
        pruned_non_zero, pruned_total = count_non_zero_params(pruned_model)
        
        print(f"Pruned model size: {pruned_size:.2f} MB")
        print(f"Pruned parameters: {pruned_params:,}")
        print(f"Pruned non-zero weights: {pruned_non_zero:,}/{pruned_total:,} ({pruned_non_zero/pruned_total*100:.2f}%)")
        
        # Calculate improvements
        size_reduction = (baseline_size - pruned_size) / baseline_size * 100
        param_reduction = (baseline_params - pruned_params) / baseline_params * 100
        sparsity = (1 - pruned_non_zero / pruned_total) * 100
        
        print(f"Size reduction: {size_reduction:.2f}%")
        print(f"Parameter reduction: {param_reduction:.2f}%")
        print(f"Model sparsity: {sparsity:.2f}%")
        
        # Save metrics
        all_metrics[model_key] = {
            "model_name": model_name,
            "task": task,
            "pruning_method": "structured",
            "pruning_amount": 0.3,
            "model_size": round(pruned_size, 2),
            "size_reduction": round(size_reduction, 2),
            "parameter_reduction": round(param_reduction, 2),
            "sparsity": round(sparsity, 2),
            "non_zero_weights": pruned_non_zero,
            "total_weights": pruned_total
        }
        
    except Exception as e:
        print(f"Error analyzing pruned model {model_key}: {e}")
        import traceback
        traceback.print_exc()

# Save all metrics to a file
metrics_path = "pruned-metrics.json"
with open(metrics_path, "w") as f:
    json.dump(all_metrics, f, indent=2)

print(f"\nSaved pruned metrics to {metrics_path}")

In [ ]:
# Create a DataFrame for comparison
comparison_data = []

for model_key in all_metrics.keys():
    metrics = all_metrics[model_key]
    
    # Prepare data for this model
    model_data = {
        'Model': metrics['model_name'],
        'Size (MB)': metrics['model_size'],
        'Size Reduction (%)': metrics['size_reduction'],
        'Sparsity (%)': metrics['sparsity']
    }
    
    comparison_data.append(model_data)

# Create DataFrame
comparison_df = pd.DataFrame(comparison_data)

# Display the DataFrame
comparison_df